# Startup Success Prediction
### Will a seed-stage startup raise a Series A?

**Data source:** Public Crunchbase investment data (2013 snapshot)
**Label:** `1` = raised Series A, `0` = did not

In [1]:
import os

os.makedirs("../data/data", exist_ok=True)
!kaggle datasets download -d arindam235/startup-investments-crunchbase -p data --unzip -q

print("Download complete. Files in data/:")
for f in os.listdir("../data/data"):
    size_kb = os.path.getsize(f"data/{f}") // 1024
    print(f"  {f}  ({size_kb} KB)")

Download complete. Files in data/:
  investments_VC.csv  (12241 KB)


In [2]:
import pandas as pd

df = pd.read_csv("data/investments_VC.csv", encoding="latin-1")
df.columns = df.columns.str.strip()

print(f"Loaded {len(df):,} companies")
print(f"\nColumns: {df.columns.tolist()}")

Loaded 54,294 companies

Columns: ['permalink', 'name', 'homepage_url', 'category_list', 'market', 'funding_total_usd', 'status', 'country_code', 'state_code', 'region', 'city', 'funding_rounds', 'founded_at', 'founded_month', 'founded_quarter', 'founded_year', 'first_funding_at', 'last_funding_at', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']


In [3]:
import numpy as np

# Keep only companies with a seed round
df_seed = df[df["seed"].notna() & (pd.to_numeric(df["seed"], errors="coerce") > 0)].copy()
df_seed["seed"] = pd.to_numeric(df_seed["seed"], errors="coerce")
df_seed["round_A"] = pd.to_numeric(df_seed.get("round_A", pd.Series(dtype=float)), errors="coerce").fillna(0)

# Binary label: did they raise a Series A?
df_seed["raised_series_a"] = (df_seed["round_A"] > 0).astype(int)

# Parse funding date for temporal splitting
df_seed["first_funding_at"] = pd.to_datetime(df_seed["first_funding_at"], errors="coerce")
df_seed = df_seed.dropna(subset=["first_funding_at"])
df_seed = df_seed.sort_values("first_funding_at")

n_yes = df_seed["raised_series_a"].sum()
n_total = len(df_seed)
print(f"Seed-stage companies with a known first-funding date: {n_total:,}")
print(f"\nClass balance:")
print(f"  Raised Series A : {n_yes:,} ({n_yes/n_total*100:.1f}%)")
print(f"  Did not raise   : {n_total - n_yes:,} ({(n_total - n_yes)/n_total*100:.1f}%)")
print(f"\nFunding date range: {df_seed['first_funding_at'].min().date()} \u2192 {df_seed['first_funding_at'].max().date()}")

Seed-stage companies with a known first-funding date: 13,839

Class balance:
  Raised Series A : 1,593 (11.5%)
  Did not raise   : 12,246 (88.5%)

Funding date range: 1921-09-01 → 2014-12-31
